In [ ]:
"""
04_unsupervised_exploration.ipynb
==================================
UMAP-based visualization of treatment effects on Cell Painting profiles.

Methods implemented:
    1. Standard UMAP (2D) — baseline visualization colored by condition/compound
    2. Semi-supervised UMAP — uses healthy/disease labels to anchor the embedding,
       treatments are unlabeled and positioned by morphological similarity
    3. 3D UMAP — interactive Plotly visualization for dose-response trajectories
    4. Parametric UMAP — neural network-based embedding for fast online projection

Each method is run on both PCA-reduced and selected-feature inputs for comparison.

Inputs:
    - outputs/01_preprocessed_features.csv
    - outputs/02_plate_selected_features.json
    - outputs/02_pca_profiles.csv
    - outputs/02_pca_models.pkl

Outputs:
    - UMAP embeddings appended to profile dataframes
    - Interactive 3D Plotly visualizations
    - Comparison plots: PCA-input vs selected-feature-input
"""

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import json
import pickle
import umap
from pathlib import Path


## Configuration & Data Loading

Two input modes for comparison:
- **PCA**: 10 principal components from notebook 03 
- **Selected**: features retained after variance/correlation/Cohen's d filtering 

In [ ]:
OUTPUT_DIR = Path("../outputs")

meta_cols = ["plate", "session", "batch", "row", "column",
             "compound", "dose_uM", "cell_type", "condition", "num_wells"]

UMAP_SEED = 42  # reproducibility

In [ ]:
# Load PCA-reduced profiles
df_pca = pd.read_csv(OUTPUT_DIR / "02_pca_profiles.csv")
pca_feature_cols = [c for c in df_pca.columns if c.startswith("PC")]

# Load selected-feature profiles
df_selected = pd.read_csv(OUTPUT_DIR / "01_preprocessed_features.csv")
with open(OUTPUT_DIR / "02_plate_selected_features.json") as f:
    plate_features = json.load(f)

# Intersection: only features retained on ALL plates
selected_feature_cols = sorted(
    set.intersection(*[set(v) for v in plate_features.values()])
)

print(f"PCA input:      {len(df_pca)} wells x {len(pca_feature_cols)} PCs")
print(f"Selected input: {len(df_selected)} wells x {len(selected_feature_cols)} features")
print(f"\nConditions (PCA df):")
print(df_pca["condition"].value_counts().to_string())

In [ ]:
# Shared plotting helpers

CONDITION_COLORS = {"healthy": "#2196F3", "disease": "#FF5722"}

def get_compound_palette(df):
    """Build a color map: fixed colors for controls, tab10 for compounds."""
    treatment_mask = ~df["condition"].isin(["healthy", "disease"])
    compounds = sorted(df.loc[treatment_mask, "compound"].unique())
    cmap = plt.cm.get_cmap("tab10", max(len(compounds), 1))
    palette = dict(CONDITION_COLORS)
    for i, cpd in enumerate(compounds):
        palette[cpd] = cmap(i)
    return palette, compounds


def plot_umap_2d(df, x_col, y_col, title, palette=None, compounds=None):
    """Scatter plot with healthy/disease as circles, treatments as diamonds."""
    if palette is None:
        palette, compounds = get_compound_palette(df)

    fig, ax = plt.subplots(figsize=(10, 7))

    for cond in ["healthy", "disease"]:
        mask = df["condition"] == cond
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   label=cond, color=palette[cond], alpha=0.7, s=60,
                   edgecolors="white", linewidth=0.5)

    treatment_mask = ~df["condition"].isin(["healthy", "disease"])
    for cpd in compounds:
        mask = (df["compound"] == cpd) & treatment_mask
        if mask.sum() == 0:
            continue
        ax.scatter(df.loc[mask, x_col], df.loc[mask, y_col],
                   label=cpd, color=palette[cpd], alpha=0.7, s=45,
                   marker="D", edgecolors="white", linewidth=0.5)

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)
    ax.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize=8)
    plt.tight_layout()
    plt.show()


def build_semi_labels(df):
    """Encode labels for semi-supervised UMAP: 0=healthy, 1=disease, -1=treatment (unlabeled)."""
    labels = np.full(len(df), -1)
    labels[df["condition"].values == "healthy"] = 0
    labels[df["condition"].values == "disease"] = 1
    return labels

---
## 1. Standard UMAP (2D)

Unsupervised UMAP knows nothing about conditions, only the feature distances between wells. 
If healthy and disease separate cleanly without labels, the morphological signal is strong.

In [ ]:
# Standard UMAP on PCA input
reducer_std = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=UMAP_SEED
)

emb_std_pca = reducer_std.fit_transform(df_pca[pca_feature_cols].values)
df_pca["UMAP1_std"] = emb_std_pca[:, 0]
df_pca["UMAP2_std"] = emb_std_pca[:, 1]

palette, compounds = get_compound_palette(df_pca)
plot_umap_2d(df_pca, "UMAP1_std", "UMAP2_std",
             "Standard UMAP \u2014 PCA Input", palette, compounds)

In [ ]:
# Standard UMAP on selected-feature input
emb_std_sel = reducer_std.fit_transform(df_selected[selected_feature_cols].values)
df_selected["UMAP1_std"] = emb_std_sel[:, 0]
df_selected["UMAP2_std"] = emb_std_sel[:, 1]

palette_sel, compounds_sel = get_compound_palette(df_selected)
plot_umap_2d(df_selected, "UMAP1_std", "UMAP2_std",
             "Standard UMAP \u2014 Selected Features Input", palette_sel, compounds_sel)

---
## 2. Semi-Supervised UMAP

Labels: `0` = healthy, `1` = disease, `-1` = treatment (unlabeled).  
UMAP uses the control labels to anchor the disease axis while letting treatments 
find their natural position based on morphological similarity.  
If a compound rescues the disease phenotype, its wells should drift toward the healthy cluster.

In [ ]:
# Semi-supervised UMAP on PCA input
labels_pca = build_semi_labels(df_pca)

reducer_semi = umap.UMAP(
    n_components=2,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=UMAP_SEED
)

emb_semi_pca = reducer_semi.fit_transform(df_pca[pca_feature_cols].values, y=labels_pca)
df_pca["UMAP1_semi"] = emb_semi_pca[:, 0]
df_pca["UMAP2_semi"] = emb_semi_pca[:, 1]

plot_umap_2d(df_pca, "UMAP1_semi", "UMAP2_semi",
             "Semi-Supervised UMAP \u2014 PCA Input", palette, compounds)

In [ ]:
# Semi-supervised UMAP on selected-feature input
labels_sel = build_semi_labels(df_selected)

emb_semi_sel = reducer_semi.fit_transform(
    df_selected[selected_feature_cols].values, y=labels_sel
)
df_selected["UMAP1_semi"] = emb_semi_sel[:, 0]
df_selected["UMAP2_semi"] = emb_semi_sel[:, 1]

plot_umap_2d(df_selected, "UMAP1_semi", "UMAP2_semi",
             "Semi-Supervised UMAP \u2014 Selected Features Input", palette_sel, compounds_sel)

---
## 3. 3D UMAP 

Three components allow dose-response trajectories to be visualized as curves through space.  
Plotly enables rotation and hover inspection to identify individual wells.

In [ ]:
# 3D UMAP on PCA input
reducer_3d = umap.UMAP(
    n_components=3,
    n_neighbors=15,
    min_dist=0.1,
    metric="euclidean",
    random_state=UMAP_SEED
)

emb_3d = reducer_3d.fit_transform(df_pca[pca_feature_cols].values)
df_pca["UMAP1_3d"] = emb_3d[:, 0]
df_pca["UMAP2_3d"] = emb_3d[:, 1]
df_pca["UMAP3_3d"] = emb_3d[:, 2]

In [ ]:
# 3D interactive plot: colored by condition/compound
df_pca["plot_label"] = df_pca["condition"]
treatment_mask = ~df_pca["condition"].isin(["healthy", "disease"])
df_pca.loc[treatment_mask, "plot_label"] = df_pca.loc[treatment_mask, "compound"]

df_pca["hover"] = (
    "plate: " + df_pca["plate"].astype(str) + "<br>" +
    "compound: " + df_pca["compound"].astype(str) + "<br>" +
    "dose: " + df_pca["dose_uM"].astype(str) + " uM<br>" +
    "condition: " + df_pca["condition"].astype(str)
)

fig = px.scatter_3d(
    df_pca,
    x="UMAP1_3d", y="UMAP2_3d", z="UMAP3_3d",
    color="plot_label",
    hover_data={"hover": True, "UMAP1_3d": False, "UMAP2_3d": False, "UMAP3_3d": False},
    title="3D UMAP \u2014 PCA Input (colored by condition/compound)",
    opacity=0.75,
    width=1000, height=700
)
fig.update_traces(marker=dict(size=4))
fig.update_layout(
    scene=dict(xaxis_title="UMAP1", yaxis_title="UMAP2", zaxis_title="UMAP3"),
    legend=dict(font=dict(size=9))
)
fig.show()

In [ ]:
# 3D plot: treatment wells only, colored by dose to show dose-response trajectories
df_treatments = df_pca[treatment_mask].copy()

if len(df_treatments) > 0 and "dose_uM" in df_treatments.columns:
    fig_dose = px.scatter_3d(
        df_treatments,
        x="UMAP1_3d", y="UMAP2_3d", z="UMAP3_3d",
        color="dose_uM",
        symbol="compound",
        hover_data={"hover": True, "UMAP1_3d": False, "UMAP2_3d": False, "UMAP3_3d": False},
        title="3D UMAP \u2014 Treatment Wells Colored by Dose",
        opacity=0.8,
        color_continuous_scale="Viridis",
        width=1000, height=700
    )
    fig_dose.update_traces(marker=dict(size=5))
    fig_dose.update_layout(
        scene=dict(xaxis_title="UMAP1", yaxis_title="UMAP2", zaxis_title="UMAP3"),
        legend=dict(font=dict(size=9))
    )
    fig_dose.show()
else:
    print("No treatment wells or dose column not found.")

---
## 4. Parametric UMAP

Replaces UMAP's SGD with a neural network, learning a parametric mapping from features to embedding.  
Once trained, new data can be embedded with a forward pass (no refitting).  

**Requires**: `pip install umap-learn[parametric]` 

In [ ]:
# Parametric UMAP on PCA input
try:
    from umap.parametric_umap import ParametricUMAP

    n_input = len(pca_feature_cols)
    reducer_param = ParametricUMAP(
        n_components=2,
        n_neighbors=15,
        min_dist=0.1,
        metric="euclidean",
        n_training_epochs=50,
        random_state=UMAP_SEED,
    )

    emb_param = reducer_param.fit_transform(df_pca[pca_feature_cols].values)
    df_pca["UMAP1_param"] = emb_param[:, 0]
    df_pca["UMAP2_param"] = emb_param[:, 1]

    plot_umap_2d(df_pca, "UMAP1_param", "UMAP2_param",
                 "Parametric UMAP \u2014 PCA Input", palette, compounds)

    # Demonstrate online embedding capability
    print("\nOnline embedding test (first 10 wells):")
    test_emb = reducer_param.transform(df_pca[pca_feature_cols].values[:10])
    print(f"  Input shape:  {df_pca[pca_feature_cols].values[:10].shape}")
    print(f"  Output shape: {test_emb.shape}")
    print("  Online embedding works \u2014 new data can be projected without refitting.")

except ImportError:
    print("Parametric UMAP not available.")
    print("Install with: pip install umap-learn[parametric]")
    print("Requires TensorFlow >= 2.0")
except Exception as e:
    print(f"Parametric UMAP failed: {e}")
    print("This is expected if TensorFlow is not configured correctly.")

---
## 5. Method Comparison

Side-by-side comparison of standard vs semi-supervised UMAP to assess how much the labels help.

In [ ]:
# Side-by-side: Standard vs Semi-supervised (PCA input)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (x_col, y_col, title) in zip(axes, [
    ("UMAP1_std", "UMAP2_std", "Standard UMAP"),
    ("UMAP1_semi", "UMAP2_semi", "Semi-Supervised UMAP")
]):
    for cond in ["healthy", "disease"]:
        mask = df_pca["condition"] == cond
        ax.scatter(df_pca.loc[mask, x_col], df_pca.loc[mask, y_col],
                   label=cond, color=palette[cond], alpha=0.7, s=50,
                   edgecolors="white", linewidth=0.5)

    tmask = ~df_pca["condition"].isin(["healthy", "disease"])
    for cpd in compounds:
        mask = (df_pca["compound"] == cpd) & tmask
        if mask.sum() == 0:
            continue
        ax.scatter(df_pca.loc[mask, x_col], df_pca.loc[mask, y_col],
                   label=cpd, color=palette[cpd], alpha=0.7, s=40,
                   marker="D", edgecolors="white", linewidth=0.5)

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)

handles, labels_ = axes[1].get_legend_handles_labels()
fig.legend(handles, labels_, bbox_to_anchor=(1.02, 0.5),
           loc="center left", fontsize=8)
plt.suptitle("Standard vs Semi-Supervised UMAP (PCA Input)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Side-by-side: PCA input vs Selected features (standard UMAP)
fig, axes = plt.subplots(1, 2, figsize=(18, 7))

for ax, (df_plot, x_col, y_col, title, pal, cpds) in zip(axes, [
    (df_pca, "UMAP1_std", "UMAP2_std",
     f"PCA Input ({len(pca_feature_cols)} PCs)", palette, compounds),
    (df_selected, "UMAP1_std", "UMAP2_std",
     f"Selected Features ({len(selected_feature_cols)} features)", palette_sel, compounds_sel)
]):
    for cond in ["healthy", "disease"]:
        mask = df_plot["condition"] == cond
        ax.scatter(df_plot.loc[mask, x_col], df_plot.loc[mask, y_col],
                   label=cond, color=pal[cond], alpha=0.7, s=50,
                   edgecolors="white", linewidth=0.5)

    tmask = ~df_plot["condition"].isin(["healthy", "disease"])
    for cpd in cpds:
        mask = (df_plot["compound"] == cpd) & tmask
        if mask.sum() == 0:
            continue
        ax.scatter(df_plot.loc[mask, x_col], df_plot.loc[mask, y_col],
                   label=cpd, color=pal[cpd], alpha=0.7, s=40,
                   marker="D", edgecolors="white", linewidth=0.5)

    ax.set_xlabel(x_col)
    ax.set_ylabel(y_col)
    ax.set_title(title)

handles, labels_ = axes[0].get_legend_handles_labels()
fig.legend(handles, labels_, bbox_to_anchor=(1.02, 0.5),
           loc="center left", fontsize=8)
plt.suptitle("Standard UMAP: PCA Input vs Selected Features", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()